<a href="https://colab.research.google.com/github/shenDivya/Capestone/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# part4_analysis.py
# Part 4: Python/Pandas Cleaning, Analysis & Cross-Validation

import pandas as pd
import numpy as np

# Load the raw CSV files
print("Loading orders_raw.csv and products.csv...\n")
orders_raw = pd.read_csv('orders_raw.csv')
products = pd.read_csv('products.csv')

# Inspect orders_raw
print("="*80)
print("ORDERS_RAW.CSV - INSPECTION")
print("="*80)

print("\n--- df.info() ---")
print(orders_raw.info())

print("\n--- df.describe() ---")
print(orders_raw.describe())

print("\n--- df['status'].value_counts() ---")
print(orders_raw['status'].value_counts())

print("\n--- First 10 rows ---")
print(orders_raw.head(10))

# Inspect products
print("\n" + "="*80)
print("PRODUCTS.CSV - INSPECTION")
print("="*80)

print("\n--- df.info() ---")
print(products.info())

print("\n--- df.describe() ---")
print(products.describe())

print("\n--- First 10 rows ---")
print(products.head(10))

# Additional inspection
print("\n" + "="*80)
print("ADDITIONAL INSPECTION")
print("="*80)

print(f"\nTotal rows in orders_raw: {len(orders_raw)}")
print(f"\nUnique cities in orders_raw:\n{orders_raw['city'].value_counts()}")
print(f"\nUnique categories in orders_raw:\n{orders_raw['category'].value_counts()}")
print(f"\nMissing values in orders_raw:\n{orders_raw.isnull().sum()}")
print(f"\nMissing values (empty strings) in amount_inr: {(orders_raw['amount_inr'] == '').sum()}")
print(f"\nSuspiciously large amount_inr values (top 10):\n{orders_raw['amount_inr'].nlargest(10)}")
# Data Quality Issues Identified

"""
1. **More than 500 rows**
- Expected: 500 rows (TOTAL_ORDERS in generate_data.py)
- Actual: 508 rows
- Issue: 8 duplicate rows were intentionally added (dup_idx in generate_data.py)

2. **Mixed casing in city column**
- Found: 'Bengaluru', 'BENGALURU', 'bengaluru', ' Bengaluru ' (with spaces)
- Found: 'Mumbai', 'MUMBAI', 'mumbai', ' Mumbai '
- Found: 'Hyderabad', 'HYDERABAD', 'hyderabad', ' Hyderabad '
- Found: 'Pune', 'PUNE', 'pune', ' Pune '
- Issue: 20 rows have inconsistent casing/spacing (casing_idx in generate_data.py)

3. **Mixed casing in category column**
- Found: 'Dairy & Eggs', 'DAIRY & EGGS', 'dairy & eggs', ' Dairy & Eggs '
- Found: Similar issues across all 6 categories
- Issue: 20 rows have inconsistent casing/spacing (variant2 in generate_data.py)

4. **Missing amount_inr values**
- Found: 10 rows with empty strings ('') in amount_inr column
- Issue: null_idx in generate_data.py set 10 rows to empty string

5. **Suspiciously large amount_inr values**
- Found: Some amount_inr values are 40x larger than normal
- Example: Values like 7200, 6400, 5600 (instead of 180, 160, 140)
- Issue: 5 rows with outliers (outlier_idx multiplied amount by 40)
"""



Loading orders_raw.csv and products.csv...

ORDERS_RAW.CSV - INSPECTION

--- df.info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       508 non-null    int64  
 1   order_date     508 non-null    object 
 2   customer_name  508 non-null    object 
 3   city           508 non-null    object 
 4   category       508 non-null    object 
 5   product_id     508 non-null    int64  
 6   quantity       508 non-null    int64  
 7   amount_inr     498 non-null    float64
 8   payment_mode   508 non-null    object 
 9   status         508 non-null    object 
 10  rating         439 non-null    float64
dtypes: float64(2), int64(3), object(6)
memory usage: 43.8+ KB
None

--- df.describe() ---
         order_id  product_id    quantity   amount_inr      rating
count  508.000000  508.000000  508.000000   498.000000  439.000000
mean   

"\n1. **More than 500 rows**\n- Expected: 500 rows (TOTAL_ORDERS in generate_data.py)\n- Actual: 508 rows\n- Issue: 8 duplicate rows were intentionally added (dup_idx in generate_data.py)\n\n2. **Mixed casing in city column**\n- Found: 'Bengaluru', 'BENGALURU', 'bengaluru', ' Bengaluru ' (with spaces)\n- Found: 'Mumbai', 'MUMBAI', 'mumbai', ' Mumbai '\n- Found: 'Hyderabad', 'HYDERABAD', 'hyderabad', ' Hyderabad '\n- Found: 'Pune', 'PUNE', 'pune', ' Pune '\n- Issue: 20 rows have inconsistent casing/spacing (casing_idx in generate_data.py)\n\n3. **Mixed casing in category column**\n- Found: 'Dairy & Eggs', 'DAIRY & EGGS', 'dairy & eggs', ' Dairy & Eggs '\n- Found: Similar issues across all 6 categories\n- Issue: 20 rows have inconsistent casing/spacing (variant2 in generate_data.py)\n\n4. **Missing amount_inr values**\n- Found: 10 rows with empty strings ('') in amount_inr column\n- Issue: null_idx in generate_data.py set 10 rows to empty string\n\n5. **Suspiciously large amount_inr valu

In [ ]:
# Load the raw CSV files
print("Loading orders_raw.csv...\n")
orders_raw = pd.read_csv('orders_raw.csv')

print("="*80)
print("STEP 1: REMOVE DUPLICATES")
print("="*80)

# Check initial row count
initial_rows = len(orders_raw)
print(f"\nInitial row count: {initial_rows}")

# Check for duplicates based on order_id
duplicate_count = orders_raw.duplicated(subset=['order_id'], keep='first').sum()
print(f"Duplicate rows found (by order_id): {duplicate_count}")

# Show duplicate order_ids
if duplicate_count > 0:
    duplicate_order_ids = orders_raw[orders_raw.duplicated(subset=['order_id'], keep=False)]['order_id'].unique()
    print(f"\nDuplicate order_ids: {sorted(duplicate_order_ids)}")

# Remove duplicates, keeping the first occurrence
orders_clean = orders_raw.drop_duplicates(subset=['order_id'], keep='first')

# Check final row count
final_rows = len(orders_clean)
rows_removed = initial_rows - final_rows

print(f"\nRows after removing duplicates: {final_rows}")
print(f"Rows removed: {rows_removed}")

# Verification
if final_rows == 500:
    print("\n✓ SUCCESS: Exactly 500 rows remain (as expected)")
else:
    print(f"\n✗ WARNING: Expected 500 rows, but got {final_rows}")

print("\n" + "="*80)

Loading orders_raw.csv...

STEP 1: REMOVE DUPLICATES

Initial row count: 508
Duplicate rows found (by order_id): 8

Duplicate order_ids: [np.int64(50), np.int64(102), np.int64(136), np.int64(139), np.int64(165), np.int64(389), np.int64(433), np.int64(440)]

Rows after removing duplicates: 500
Rows removed: 8

✓ SUCCESS: Exactly 500 rows remain (as expected)



In [ ]:
# Remove duplicates, keeping the first occurrence
initial_rows = len(orders_raw)
orders_clean = orders_raw.drop_duplicates(subset=['order_id'], keep='first')
final_rows = len(orders_clean)

print(f"Initial rows: {initial_rows}")
print(f"Rows removed: {initial_rows - final_rows}")
print(f"Final rows: {final_rows}")

print("\n" + "="*80)
print("STEP 2: FIX CASING AND WHITESPACE")
print("="*80)

# Show distinct values BEFORE cleaning
print("\n--- BEFORE Cleaning ---")
print(f"\nDistinct cities: {orders_clean['city'].nunique()}")
print(orders_clean['city'].unique())

print(f"\nDistinct categories: {orders_clean['category'].nunique()}")
print(orders_clean['category'].unique())

# Clean city column: strip whitespace and apply title case
orders_clean['city'] = orders_clean['city'].str.strip().str.title()

# Clean category column: strip whitespace and apply title case
orders_clean['category'] = orders_clean['category'].str.strip().str.title()

# Show distinct values AFTER cleaning
print("\n--- AFTER Cleaning ---")
print(f"\nDistinct cities: {orders_clean['city'].nunique()}")
print("City values:")
for city in sorted(orders_clean['city'].unique()):
    print(f"  - {city}")

print(f"\nDistinct categories: {orders_clean['category'].nunique()}")
print("Category values:")
for category in sorted(orders_clean['category'].unique()):
    print(f"  - {category}")

# Verification
if orders_clean['city'].nunique() == 4:
    print("\n✓ City column has exactly 4 distinct values")
else:
    print(f"\n✗ WARNING: Expected 4 distinct cities, got {orders_clean['city'].nunique()}")

if orders_clean['category'].nunique() == 6:
    print("✓ Category column has exactly 6 distinct values")
else:
    print(f"✗ WARNING: Expected 6 distinct categories, got {orders_clean['category'].nunique()}")

print("\n" + "="*80)

Initial rows: 508
Rows removed: 8
Final rows: 500

STEP 2: FIX CASING AND WHITESPACE

--- BEFORE Cleaning ---

Distinct cities: 14
['Pune' 'Bengaluru' 'Hyderabad' 'Mumbai' ' Bengaluru ' 'BENGALURU'
 ' Mumbai ' 'HYDERABAD' 'mumbai' 'MUMBAI' ' Pune ' ' Hyderabad '
 'bengaluru' 'PUNE']

Distinct categories: 18
['Snacks & Beverages' 'Household Essentials' 'Bakery' 'Personal Care'
 'Fruits & Vegetables' 'Dairy & Eggs' ' Dairy & Eggs ' 'BAKERY'
 ' Household Essentials ' 'DAIRY & EGGS' ' Bakery ' 'SNACKS & BEVERAGES'
 'dairy & eggs' 'HOUSEHOLD ESSENTIALS' ' Personal Care '
 'FRUITS & VEGETABLES' ' Fruits & Vegetables ' 'snacks & beverages']

--- AFTER Cleaning ---

Distinct cities: 4
City values:
  - Bengaluru
  - Hyderabad
  - Mumbai
  - Pune

Distinct categories: 6
Category values:
  - Bakery
  - Dairy & Eggs
  - Fruits & Vegetables
  - Household Essentials
  - Personal Care
  - Snacks & Beverages

✓ City column has exactly 4 distinct values
✓ Category column has exactly 6 distinct values



/tmp/ipykernel_3833/3997878962.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean['city'] = orders_clean['city'].str.strip().str.title()
/tmp/ipykernel_3833/3997878962.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean['category'] = orders_clean['category'].str.strip().str.title()


In [ ]:
# Check for missing values in amount_inr
# Note: amount_inr was set to empty string ('') not NaN in generate_data.py
print("\n--- Missing amount_inr Analysis ---")

# Convert amount_inr to numeric, treating empty strings and non-numeric as NaN
orders_clean['amount_inr'] = pd.to_numeric(orders_clean['amount_inr'], errors='coerce')

missing_amount_count = orders_clean['amount_inr'].isna().sum()
total_rows = len(orders_clean)

print(f"\nTotal rows: {total_rows}")
print(f"Missing amount_inr values: {missing_amount_count}")
print(f"Percentage of missing values: {(missing_amount_count/total_rows)*100:.2f}%")

# Show some examples of rows with missing amount_inr
if missing_amount_count > 0:
    print(f"\nSample rows with missing amount_inr:")
    print(orders_clean[orders_clean['amount_inr'].isna()][['order_id', 'customer_name', 'city', 'category', 'amount_inr', 'status']].head())

print("\n--- Rating Column Analysis ---")

# Check rating column
rating_null_count = orders_clean['rating'].isna().sum()
rating_null_by_status = orders_clean.groupby('status')['rating'].apply(lambda x: x.isna().sum())

print(f"\nTotal null ratings: {rating_null_count}")
print(f"\nNull ratings by status:")
print(rating_null_by_status)

# Verify that all Cancelled and Pending orders have null ratings
cancelled_pending = orders_clean[orders_clean['status'].isin(['Cancelled', 'Pending'])]
cancelled_pending_with_null_rating = cancelled_pending['rating'].isna().sum()

print(f"\nCancelled + Pending orders: {len(cancelled_pending)}")
print(f"Cancelled + Pending orders with null rating: {cancelled_pending_with_null_rating}")

if cancelled_pending_with_null_rating == len(cancelled_pending):
    print("✓ All Cancelled and Pending orders have null ratings (as expected)")
else:
    print("✗ WARNING: Some Cancelled/Pending orders have ratings")

# Check Delivered orders
delivered = orders_clean[orders_clean['status'] == 'Delivered']
delivered_with_rating = delivered['rating'].notna().sum()
delivered_total = len(delivered)

print(f"\nDelivered orders: {delivered_total}")
print(f"Delivered orders with rating: {delivered_with_rating}")
print(f"Delivered orders with null rating: {delivered['rating'].isna().sum()}")

print("\n--- Business Logic for Missing Values ---")
print("\n1. amount_inr:")
print(f"   - {missing_amount_count} rows have missing amount_inr values")
print("   - These rows will be EXCLUDED from all revenue calculations")
print("   - Rationale: Missing revenue is unknown, not zero")
print("   - Filling with 0 or mean would silently distort totals")

print("\n2. rating:")
print(f"   - {rating_null_count} rows have null ratings")
print("   - These are legitimately null (Cancelled and Pending orders)")
print("   - These nulls will be LEFT AS-IS")
print("   - Rationale: Only Delivered orders receive ratings")
print("   - This is NOT a data quality problem")

# Create a clean dataset for revenue calculations (excluding missing amount_inr)
orders_for_revenue = orders_clean[orders_clean['amount_inr'].notna()].copy()

print(f"\n--- Dataset Summary ---")
print(f"Total rows after cleaning: {len(orders_clean)}")
print(f"Rows available for revenue calculations: {len(orders_for_revenue)}")
print(f"Rows excluded from revenue calculations: {len(orders_clean) - len(orders_for_revenue)}")

print("\n" + "="*80)


--- Missing amount_inr Analysis ---

Total rows: 500
Missing amount_inr values: 10
Percentage of missing values: 2.00%

Sample rows with missing amount_inr:
     order_id customer_name       city              category  amount_inr  \
33         34          Myra  Bengaluru          Dairy & Eggs         NaN   
35         36          Riya       Pune         Personal Care         NaN   
113       114        Shreya       Pune  Household Essentials         NaN   
231       232         Divya       Pune         Personal Care         NaN   
268       269        Nikhil       Pune         Personal Care         NaN   

        status  
33   Delivered  
35   Delivered  
113    Pending  
231  Delivered  
268  Delivered  

--- Rating Column Analysis ---

Total null ratings: 66

Null ratings by status:
status
Cancelled    42
Delivered     0
Pending      24
Name: rating, dtype: int64

Cancelled + Pending orders: 66
Cancelled + Pending orders with null rating: 66
✓ All Cancelled and Pending orders have 

/tmp/ipykernel_3833/1254237080.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean['amount_inr'] = pd.to_numeric(orders_clean['amount_inr'], errors='coerce')


In [ ]:
print("\n" + "="*80)
print("STEP 3: HANDLE MISSING VALUES")
print("="*80)

# Check for missing values in amount_inr
# Note: amount_inr was set to empty string ('') not NaN in generate_data.py
print("\n--- Missing amount_inr Analysis ---")

# Convert amount_inr to numeric, treating empty strings and non-numeric as NaN
orders_clean['amount_inr'] = pd.to_numeric(orders_clean['amount_inr'], errors='coerce')

missing_amount_count = orders_clean['amount_inr'].isna().sum()
total_rows = len(orders_clean)

print(f"\nTotal rows: {total_rows}")
print(f"Missing amount_inr values: {missing_amount_count}")
print(f"Percentage of missing values: {(missing_amount_count/total_rows)*100:.2f}%")

# Show some examples of rows with missing amount_inr
if missing_amount_count > 0:
    print(f"\nSample rows with missing amount_inr:")
    print(orders_clean[orders_clean['amount_inr'].isna()][['order_id', 'customer_name', 'city', 'category', 'amount_inr', 'status']].head())

print("\n--- Rating Column Analysis ---")

# Check rating column
rating_null_count = orders_clean['rating'].isna().sum()
rating_null_by_status = orders_clean.groupby('status')['rating'].apply(lambda x: x.isna().sum())

print(f"\nTotal null ratings: {rating_null_count}")
print(f"\nNull ratings by status:")
print(rating_null_by_status)

# Verify that all Cancelled and Pending orders have null ratings
cancelled_pending = orders_clean[orders_clean['status'].isin(['Cancelled', 'Pending'])]
cancelled_pending_with_null_rating = cancelled_pending['rating'].isna().sum()

print(f"\nCancelled + Pending orders: {len(cancelled_pending)}")
print(f"Cancelled + Pending orders with null rating: {cancelled_pending_with_null_rating}")

if cancelled_pending_with_null_rating == len(cancelled_pending):
    print("✓ All Cancelled and Pending orders have null ratings (as expected)")
else:
    print("✗ WARNING: Some Cancelled/Pending orders have ratings")

# Check Delivered orders
delivered = orders_clean[orders_clean['status'] == 'Delivered']
delivered_with_rating = delivered['rating'].notna().sum()
delivered_total = len(delivered)

print(f"\nDelivered orders: {delivered_total}")
print(f"Delivered orders with rating: {delivered_with_rating}")
print(f"Delivered orders with null rating: {delivered['rating'].isna().sum()}")

print("\n--- Business Logic for Missing Values ---")
print("\n1. amount_inr:")
print(f"   - {missing_amount_count} rows have missing amount_inr values")
print("   - These rows will be EXCLUDED from all revenue calculations")
print("   - Rationale: Missing revenue is unknown, not zero")
print("   - Filling with 0 or mean would silently distort totals")

print("\n2. rating:")
print(f"   - {rating_null_count} rows have null ratings")
print("   - These are legitimately null (Cancelled and Pending orders)")
print("   - These nulls will be LEFT AS-IS")
print("   - Rationale: Only Delivered orders receive ratings")
print("   - This is NOT a data quality problem")

# Create a clean dataset for revenue calculations (excluding missing amount_inr)
orders_for_revenue = orders_clean[orders_clean['amount_inr'].notna()].copy()

print(f"\n--- Dataset Summary ---")
print(f"Total rows after cleaning: {len(orders_clean)}")
print(f"Rows available for revenue calculations: {len(orders_for_revenue)}")
print(f"Rows excluded from revenue calculations: {len(orders_clean) - len(orders_for_revenue)}")

print("\n" + "="*80)


STEP 3: HANDLE MISSING VALUES

--- Missing amount_inr Analysis ---

Total rows: 500
Missing amount_inr values: 10
Percentage of missing values: 2.00%

Sample rows with missing amount_inr:
     order_id customer_name       city              category  amount_inr  \
33         34          Myra  Bengaluru          Dairy & Eggs         NaN   
35         36          Riya       Pune         Personal Care         NaN   
113       114        Shreya       Pune  Household Essentials         NaN   
231       232         Divya       Pune         Personal Care         NaN   
268       269        Nikhil       Pune         Personal Care         NaN   

        status  
33   Delivered  
35   Delivered  
113    Pending  
231  Delivered  
268  Delivered  

--- Rating Column Analysis ---

Total null ratings: 66

Null ratings by status:
status
Cancelled    42
Delivered     0
Pending      24
Name: rating, dtype: int64

Cancelled + Pending orders: 66
Cancelled + Pending orders with null rating: 66
✓ All Can

/tmp/ipykernel_3833/1433065028.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean['amount_inr'] = pd.to_numeric(orders_clean['amount_inr'], errors='coerce')


In [ ]:
print("\n" + "="*80)
print("STEP 4: DETECT AND CAP OUTLIERS WITH IQR")
print("="*80)

# Filter to Delivered orders with non-null amount_inr only
delivered_orders = orders_for_revenue[orders_for_revenue['status'] == 'Delivered'].copy()
print(f"\nDelivered orders with non-null amount_inr: {len(delivered_orders)}")

# Compute quartiles and IQR
Q1 = delivered_orders['amount_inr'].quantile(0.25)
Q3 = delivered_orders['amount_inr'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

print(f"\nComputed statistics:")
print(f"  Q1 (25th percentile): {Q1:.2f} INR")
print(f"  Q3 (75th percentile): {Q3:.2f} INR")
print(f"  IQR: {IQR:.2f} INR")
print(f"  Upper fence: {upper_fence:.2f} INR")

# Identify outliers (values above upper fence)
outlier_mask = delivered_orders['amount_inr'] > upper_fence
outlier_count = outlier_mask.sum()
print(f"\nOutliers detected (amount_inr > {upper_fence:.2f}): {outlier_count} rows")

# Cap these values using .clip(upper=upper_fence)
# We create a new column 'amount_inr_capped' in the delivered_orders subset
delivered_orders.loc[:, 'amount_inr_capped'] = delivered_orders['amount_inr'].clip(upper=upper_fence)

# Show how many rows were actually capped (values changed)
capped_rows = delivered_orders.loc[outlier_mask, 'amount_inr_capped'] != delivered_orders.loc[outlier_mask, 'amount_inr']
actually_capped = capped_rows.sum()
print(f"Rows actually capped (value > upper_fence): {actually_capped}")

# Update the main orders_for_revenue DataFrame with capped values for Delivered rows
orders_for_revenue = orders_for_revenue.merge(
    delivered_orders[['order_id', 'amount_inr_capped']],
    on='order_id',
    how='left'
)
# Replace amount_inr with capped value where applicable, else keep original orders_for_revenue['amount_inr_capped'] = orders_for_revenue['amount_inr_capped'].fillna(orders_for_revenue['amount_inr'])

# For clarity, we can now set amount_inr = amount_inr_capped if we want a single cleaned column.
# But to preserve original, we'll keep both. The analysis tasks will use 'amount_inr_capped' as final revenue.

print(f"\nCapping complete. {actually_capped} rows were capped from their original value to {upper_fence:.2f} INR.")
print(f"Note: This includes both synthetic outliers and some real high-value orders (expected).")

print("\n" + "="*80)
print("CLEANING COMPLETE")
print("="*80)
print(f"Rows in final cleaned dataset (orders_clean): {len(orders_clean)}")
print(f"Rows in revenue dataset with capped values (orders_for_revenue): {len(orders_for_revenue)}")



STEP 4: DETECT AND CAP OUTLIERS WITH IQR

Delivered orders with non-null amount_inr: 425

Computed statistics:
  Q1 (25th percentile): 90.00 INR
  Q3 (75th percentile): 275.00 INR
  IQR: 185.00 INR
  Upper fence: 552.50 INR

Outliers detected (amount_inr > 552.50): 16 rows
Rows actually capped (value > upper_fence): 16

Capping complete. 16 rows were capped from their original value to 552.50 INR.
Note: This includes both synthetic outliers and some real high-value orders (expected).

CLEANING COMPLETE
Rows in final cleaned dataset (orders_clean): 500
Rows in revenue dataset with capped values (orders_for_revenue): 490


In [ ]:
print("\n" + "="*80)
print("STEP 5: PARSE DATES AND CREATE DERIVED COLUMNS")
print("="*80)

# Parse order_date to datetime
print("\n--- Converting order_date to datetime ---")
orders_for_revenue['order_date'] = pd.to_datetime(orders_for_revenue['order_date'])

print(f"order_date dtype: {orders_for_revenue['order_date'].dtype}")
print(f"Sample dates:\n{orders_for_revenue['order_date'].head()}")

# Extract month (numeric) and month_name
print("\n--- Extracting month and month_name ---")
orders_for_revenue['month'] = orders_for_revenue['order_date'].dt.month
orders_for_revenue['month_name'] = orders_for_revenue['order_date'].dt.month_name()

print(f"\nSample month values:\n{orders_for_revenue[['order_date', 'month', 'month_name']].head(10)}")
print(f"\nDistinct months: {sorted(orders_for_revenue['month'].unique())}")
print(f"Distinct month names: {sorted(orders_for_revenue['month_name'].unique())}")

# Create revenue_per_unit using capped amount
print("\n--- Creating revenue_per_unit ---")
orders_for_revenue['revenue_per_unit'] = orders_for_revenue['amount_inr_capped'] / orders_for_revenue['quantity']

print(f"\nSample revenue_per_unit calculations:")
print(orders_for_revenue[['order_id', 'amount_inr_capped', 'quantity', 'revenue_per_unit']].head(10))

# Create is_delivered boolean column
print("\n--- Creating is_delivered boolean ---")
orders_for_revenue['is_delivered'] = orders_for_revenue['status'] == 'Delivered'

print(f"\nValue counts for is_delivered:")
print(orders_for_revenue['is_delivered'].value_counts())

# Verify against status
print(f"\nStatus breakdown:")
print(orders_for_revenue['status'].value_counts())

print("\n" + "="*80)
print("CLEANING AND FEATURE ENGINEERING COMPLETE")
print("="*80)

print(f"\nFinal dataset summary:")
print(f"  Total rows: {len(orders_for_revenue)}")
print(f"  Columns: {list(orders_for_revenue.columns)}")
print(f"\nNew derived columns created:")
print(f"  - month (numeric): {orders_for_revenue['month'].nunique()} unique values")
print(f"  - month_name (string): {orders_for_revenue['month_name'].nunique()} unique values")
print(f"  - revenue_per_unit (float): calculated from capped amount / quantity")
print(f"  - is_delivered (boolean): True for Delivered orders")

print("\n" + "="*80)


STEP 5: PARSE DATES AND CREATE DERIVED COLUMNS

--- Converting order_date to datetime ---
order_date dtype: datetime64[ns]
Sample dates:
0   2026-03-09
1   2026-05-22
2   2026-06-30
3   2026-03-01
4   2026-04-05
Name: order_date, dtype: datetime64[ns]

--- Extracting month and month_name ---

Sample month values:
  order_date  month month_name
0 2026-03-09      3      March
1 2026-05-22      5        May
2 2026-06-30      6       June
3 2026-03-01      3      March
4 2026-04-05      4      April
5 2026-06-05      6       June
6 2026-03-11      3      March
7 2026-01-09      1    January
8 2026-02-24      2   February
9 2026-03-05      3      March

Distinct months: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]
Distinct month names: ['April', 'February', 'January', 'June', 'March', 'May']

--- Creating revenue_per_unit ---

Sample revenue_per_unit calculations:
   order_id  amount_inr_capped  quantity  revenue_per_unit
0         1              175.0    

In [ ]:
print("\n" + "="*80)
print("STEP 6: GROUP, MERGE, AND ANSWER BUSINESS QUESTIONS")
print("="*80)

# ---------------------------------------------------------------------------
# (a) Total revenue per category (Delivered only, cleaned + capped data)
# ---------------------------------------------------------------------------
print("\n--- (a) Total revenue per category (Delivered only) ---")

delivered_clean = orders_for_revenue[orders_for_revenue['is_delivered']].copy()

revenue_by_category = (
    delivered_clean
    .groupby('category')['amount_inr_capped']
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_category)

top_category = revenue_by_category.index[0]
top_category_revenue = revenue_by_category.iloc[0]
print(f"\nTop category by revenue: {top_category} ({top_category_revenue:,.2f} INR)")

# ---------------------------------------------------------------------------
# (b) Merge with products.csv to bring in supplier, then group by supplier
# ---------------------------------------------------------------------------
print("\n--- (b) Revenue by supplier (merged with products.csv) ---")

products = pd.read_csv('products.csv')

orders_with_supplier = pd.merge(
    delivered_clean,
    products[['product_id', 'product_name', 'supplier']],
    on='product_id',
    how='left'
)

# Sanity check: no rows lost or gained in the merge
print(f"Rows before merge: {len(delivered_clean)}")
print(f"Rows after merge:  {len(orders_with_supplier)}")
print(f"Rows with missing supplier after merge: {orders_with_supplier['supplier'].isna().sum()}")

revenue_by_supplier = (
    orders_with_supplier
    .groupby('supplier')['amount_inr_capped']
    .sum()
    .sort_values(ascending=False)
)
print("\n", revenue_by_supplier)

top_supplier = revenue_by_supplier.index[0]
top_supplier_revenue = revenue_by_supplier.iloc[0]
print(f"\nTop supplier by revenue: {top_supplier} ({top_supplier_revenue:,.2f} INR)")

# ---------------------------------------------------------------------------
# (c) Cross-validate against Part 1 SQL diagnostic
# ---------------------------------------------------------------------------
print("\n--- (c) Cross-validation against Part 1 ---")

PART1_TOP_CATEGORY = 'Household Essentials'
PART1_TOP_SUPPLIER = 'HomeEssentials Traders'

category_match = (top_category == PART1_TOP_CATEGORY)
supplier_match = (top_supplier == PART1_TOP_SUPPLIER)

print(f"Part 4 top category: {top_category}  | Part 1 top category: {PART1_TOP_CATEGORY}  -> "
      f"{'MATCH' if category_match else 'MISMATCH'}")
print(f"Part 4 top supplier: {top_supplier}  | Part 1 top supplier: {PART1_TOP_SUPPLIER}  -> "
      f"{'MATCH' if supplier_match else 'MISMATCH'}")

if category_match and supplier_match:
    print("\n✓ Cross-validation passed: top category and top supplier match Part 1.")
    print("  Note: exact rupee totals differ from Part 1 because Part 4 started from dirtier data")
    print("  (duplicates removed, 10 missing amount_inr rows excluded, outliers capped).")
else:
    print("\n✗ Cross-validation FAILED: check the cleaning pipeline (duplicates, casing, "
          "missing-value exclusion, capping) before submitting.")

print("\n" + "="*80)



STEP 6: GROUP, MERGE, AND ANSWER BUSINESS QUESTIONS

--- (a) Total revenue per category (Delivered only) ---
category
Household Essentials    20910.0
Bakery                  14975.0
Personal Care           14594.5
Dairy & Eggs            13675.0
Snacks & Beverages      11312.5
Fruits & Vegetables      9170.0
Name: amount_inr_capped, dtype: float64

Top category by revenue: Household Essentials (20,910.00 INR)

--- (b) Revenue by supplier (merged with products.csv) ---
Rows before merge: 425
Rows after merge:  425
Rows with missing supplier after merge: 0

 supplier
HomeEssentials Traders    20910.0
BakeHouse Supplies        18550.0
CarePlus Distributors     14594.5
DairyBest Ltd             10710.0
SnackHub India             7737.5
FreshFarms Co              5490.0
GreenValley Traders        3680.0
CountryEggs Farms          2965.0
Name: amount_inr_capped, dtype: float64

Top supplier by revenue: HomeEssentials Traders (20,910.00 INR)

--- (c) Cross-validation against Part 1 ---
Part 

In [ ]:

import matplotlib
matplotlib.use('Agg')  # write PNG files without needing a display
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("STEP 7: VISUALIZE")
print("="*80)

# ---------------------------------------------------------------------------
# Chart 1: Bar chart of category revenue (post-cleaning, Delivered only)
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d9534f' if cat == top_category else '#5bc0de' for cat in revenue_by_category.index]
ax.bar(revenue_by_category.index, revenue_by_category.values, color=colors)
ax.set_title(f"{top_category} is the top revenue category post-cleaning "
             f"({top_category_revenue:,.0f} INR, Delivered orders only)")
ax.set_xlabel("Category")
ax.set_ylabel("Total revenue (INR, capped)")
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(revenue_by_category.values):
    ax.text(i, v, f"{v:,.0f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('chart1_category_revenue.png', dpi=150)
plt.close()
print("Saved chart1_category_revenue.png")

# ---------------------------------------------------------------------------
# Chart 2: Line chart of total monthly revenue trend (Delivered only)
# ---------------------------------------------------------------------------
monthly_revenue = (
    delivered_clean
    .groupby(['month', 'month_name'])['amount_inr_capped']
    .sum()
    .reset_index()
    .sort_values('month')
)
peak_row = monthly_revenue.loc[monthly_revenue['amount_inr_capped'].idxmax()]
low_row = monthly_revenue.loc[monthly_revenue['amount_inr_capped'].idxmin()]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(monthly_revenue['month_name'], monthly_revenue['amount_inr_capped'],
        marker='o', linewidth=2, color='#0275d8')
ax.set_title(f"Monthly Delivered revenue peaks in {peak_row['month_name']} "
             f"({peak_row['amount_inr_capped']:,.0f} INR) and is lowest in {low_row['month_name']} "
             f"({low_row['amount_inr_capped']:,.0f} INR)")
ax.set_xlabel("Month (2026)")
ax.set_ylabel("Total revenue (INR, capped)")
ax.grid(True, alpha=0.3)
for x, y in zip(monthly_revenue['month_name'], monthly_revenue['amount_inr_capped']):
    ax.annotate(f"{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('chart2_monthly_revenue_trend.png', dpi=150)
plt.close()
print("Saved chart2_monthly_revenue_trend.png")
print(monthly_revenue[['month_name', 'amount_inr_capped']].to_string(index=False))

# ---------------------------------------------------------------------------
# Chart 3 (choice): Horizontal bar chart of revenue by supplier
# Supports the insight that HomeEssentials Traders dominates supplier revenue
# ---------------------------------------------------------------------------
supplier_sorted = revenue_by_supplier.sort_values(ascending=True)
share = top_supplier_revenue / revenue_by_supplier.sum() * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d9534f' if s == top_supplier else '#5cb85c' for s in supplier_sorted.index]
ax.barh(supplier_sorted.index, supplier_sorted.values, color=colors)
ax.set_title(f"{top_supplier} generates the most revenue "
             f"({top_supplier_revenue:,.0f} INR, {share:.1f}% of Delivered revenue)")
ax.set_xlabel("Total revenue (INR, capped)")
ax.set_ylabel("Supplier")
for i, v in enumerate(supplier_sorted.values):
    ax.text(v, i, f" {v:,.0f}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig('chart3_supplier_revenue.png', dpi=150)
plt.close()
print("Saved chart3_supplier_revenue.png")

print("\n" + "="*80)


STEP 7: VISUALIZE
Saved chart1_category_revenue.png
Saved chart2_monthly_revenue_trend.png
month_name  amount_inr_capped
   January            10059.5
  February            13715.0
     March            14747.5
     April            16558.5
       May            16668.5
      June            12888.0
Saved chart3_supplier_revenue.png



## Observation 1 — Household Essentials is the single largest revenue category

**What:** After deduplication (508 → 500 rows), exclusion of the 10 rows with unknown
`amount_inr`, and IQR capping at the upper fence (Q3 + 1.5 × IQR), `groupby('category')`
on Delivered orders shows **Household Essentials as the top category by revenue**, ahead
of all five other categories. This ranking is identical to Part 1's clean SQL diagnostic,
even though the rupee totals differ because Part 4 removed duplicates, dropped unknown
amounts, and clipped high-value orders.

**Why it matters:** A single category carrying the top revenue position means stock-outs,
price changes, or delivery failures in Household Essentials hit the P&L harder than an
equivalent problem in any other category. It is also the category that most influences
whether the overall revenue target is met.

**Next step:** Protect this category first — review reorder points and safety stock for
the Household Essentials SKUs, and check whether its lead comes from high order volume or
high average order value before deciding between a volume push or a margin push.

---

## Observation 2 — Revenue is concentrated in one supplier

**What:** After merging the cleaned Delivered orders with `products.csv` on `product_id`
and grouping by `supplier`, **HomeEssentials Traders is the top supplier by revenue**, and
the horizontal bar chart shows its share is materially larger than the next supplier's.
This matches Part 1's finding that HomeEssentials Traders is the dominant supplier.

**Why it matters:** Revenue concentrated in one supplier is a continuity risk. A price
increase, a quality issue, or a delivery disruption from that one vendor propagates
directly into the top revenue category, because the top supplier and top category are the
same business line.

**Next step:** Quantify the dependency precisely (supplier revenue ÷ total delivered
revenue) and either negotiate a fixed-price contract with HomeEssentials Traders or
onboard a second supplier for the highest-revenue SKUs in that category.

---

## Observation 3 — Roughly 13% of raw order rows were unusable or distorted before cleaning

**What:** The raw export contained **508 rows**, of which **8 were duplicate `order_id`s**,
**10 had missing `amount_inr`**, **20 had inconsistent city/category casing or whitespace**
(city collapsed from 12 → 4 distinct values, category from 18 → 6), and **~16 Delivered rows
exceeded the IQR upper fence** and were capped. Without cleaning, category and supplier
totals would have been inflated by double-counted rows and by the extreme `amount_inr`
values.

**Why it matters:** Every downstream number — category revenue, supplier ranking, monthly
trend — is computed from these fields. Duplicates inflate revenue, unstandardised casing
splits one category into several in a `groupby`, and extreme values pull averages away from
typical order behaviour. Reporting on the raw file would have produced wrong totals and a
fragmented category list.

**Next step:** Push the fixes upstream rather than repeating them in every analysis — add a
unique constraint on `order_id` at the export layer, make `amount_inr` a required field, and
validate `city`/`category` against a controlled list so the export arrives analysis-ready.
